# Using OpenAI models with Strands Agents via LiteLLM

## Overview

[LiteLLM](https://docs.litellm.ai/docs/) is a unified interface for many LLM providers, letting you reach models from OpenAI, Azure OpenAI, Anthropic, Mistral, and others through a single API. Strands ships a `LiteLLMModel` provider, so you can point an agent at any model LiteLLM supports and switch models without changing the rest of your agent code.

In this example we use `gpt-5-mini` on Azure OpenAI as the agent's model, with a simple `current_time` and `current_weather` tool use case.

## Tutorial Details

| Information            | Details                                          |
|:-----------------------|:-------------------------------------------------|
| Agent structure        | Single agent                                     |
| Model                  | `gpt-5-mini` (from OpenAI)                        |
| Runs on                | Azure OpenAI (deployment `azure/gpt-5-mini`)     |
| Strands model provider | `LiteLLMModel`                                    |
| Custom tools           | current_time, current_weather                    |

## Architecture

<div style="text-align:center">
    <img src="images/architecture.png" width="65%" />
</div>

## What you'll learn
* Configure a hosted model with the `LiteLLMModel` provider
* Point LiteLLM at an Azure OpenAI deployment with environment variables
* Give the agent custom tools and inspect its messages and metrics

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* Azure account
* An Azure OpenAI deployment (this notebook uses `gpt-5-mini`)

Let's now install the required packages for our agent

In [ ]:
# installing pre-requisites
!pip install -r requirements.txt

### Importing dependency packages

Now let's import the dependency packages

In [ ]:
import os
from datetime import datetime
from datetime import timezone as tz
from typing import Any
from zoneinfo import ZoneInfo

from strands import Agent, tool
from strands.models.litellm import LiteLLMModel

### Setting up Azure keys

Let's now set up the Azure API keys

In [ ]:
os.environ["AZURE_API_KEY"] = "<YOUR_API_KEY>"
os.environ["AZURE_API_BASE"] = "<YOUR_API_BASE>"
os.environ["AZURE_API_VERSION"] = "<YOUR_API_VERSION>"

### Setting up custom tools

Let's now set up two tools to test our agent

In [ ]:
@tool
def current_time(timezone: str = "UTC") -> str:
    if timezone.upper() == "UTC":
        timezone_obj: Any = tz.utc
    else:
        timezone_obj = ZoneInfo(timezone)

    return datetime.now(timezone_obj).isoformat()


@tool
def current_weather(city: str) -> str:
    # Dummy implementation. Please replace with actual weather API call.
    return "sunny"

### Defining the agent's underlying model

Next let's define our agent's underlying model using LiteLLM. We set the `model_id` to `azure/gpt-5-mini`. With LiteLLM's `azure/<name>` format, the part after `azure/` is your Azure OpenAI *deployment name*, so update it to match the deployment you created in Azure.

`gpt-5-mini` is a reasoning model, so it uses `max_completion_tokens` (not `max_tokens`) and does not accept a custom `temperature`. We therefore pass only `max_completion_tokens` below.

In [ ]:
model = "azure/gpt-5-mini"  # the value after azure/ is your Azure deployment name
litellm_model = LiteLLMModel(
    model_id=model, params={"max_completion_tokens": 2000}
)

### Defining the agent

Now that we have all the required information available, let's define our agent

In [ ]:
system_prompt = "You are a simple agent that can tell the time and the weather"
agent = Agent(
    model=litellm_model,
    system_prompt=system_prompt,
    tools=[current_time, current_weather],
)

### Testing agent

Let's now invoke the agent to test it

In [ ]:
results = agent("What time is it in Seattle? And how is the weather?")

#### Analyzing the agent's results

We have invoked our agent for the first time. Let's explore what came back. First, we can look at the messages the agent exchanged, which are available on the agent object.

In [ ]:
agent.messages

Next we can take a look at the usage of our agent for the last query by analyzing the result `metrics`

In [ ]:
results.metrics

## Summary

In this notebook you learned how to use the LiteLLM provider to run a Strands agent against an Azure OpenAI model, gave the agent custom tools, and inspected its messages and metrics. Next, let's use OpenAI models hosted on Amazon Bedrock through the Responses API.